# Attention & PAA Attribution Gallery

This notebook provides an interpretability gallery for the trained StrokeGAT model,
showing how attention mechanisms and **Propagation-based Attention Attribution (PAA)**
reveal which brain regions drive stroke detection predictions.

## PAA Overview (Eq. 12)

PAA computes attribution scores by propagating attention weights through the
multi-layer GAT, accounting for how information flows from input nodes to
output predictions:

- **Layer-wise attention:** Each GAT layer produces attention coefficients
  between connected nodes (Eq. 2-3)
- **Multi-head aggregation:** Attention from all heads is averaged per layer
- **Cross-layer propagation:** Attention matrices are multiplied across layers
  to obtain end-to-end attribution scores
- **Edge attribution:** Final PAA scores indicate how much each edge
  contributed to the classification of each node

This provides clinically interpretable maps showing which anatomical regions
and tissue connections are most informative for stroke detection.

In [ ]:
%matplotlib inline

import sys
sys.path.insert(0, "../src")

import torch
import numpy as np
import matplotlib.pyplot as plt

from stroke_gat.config import Config
from stroke_gat.models.gat import StrokeGAT
from stroke_gat.training.callbacks import AttentionExtractionCallback
from stroke_gat.visualization.attention_maps import (
    plot_attention_by_layer,
    plot_attention_by_region,
)
from stroke_gat.visualization.attribution import (
    plot_paa_attribution_map,
    plot_stroke_penumbra_detection,
)
from stroke_gat.visualization.graph_3d import plot_graph_3d_interactive

print("Imports successful.")

In [ ]:
# Load configuration and trained model from checkpoint
cfg = Config.from_yaml("../configs/default.yaml")

model = StrokeGAT(
    in_channels=cfg.model.in_channels,
    hidden_channels=cfg.model.hidden_channels,
    out_channels=cfg.model.num_classes,
    num_layers=cfg.model.num_layers,
    num_heads=cfg.model.num_heads,
    dropout=cfg.model.dropout,
)

# Load best checkpoint
checkpoint_path = cfg.training.checkpoint_dir / "best_model.pt"
checkpoint = torch.load(checkpoint_path, map_location="cpu")
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")
print(f"Best validation score: {checkpoint['best_score']:.4f}")

In [ ]:
# Load a test graph and run inference with attention extraction
from stroke_gat.data.service import DataService
from stroke_gat.graph.builder import GraphBuilder

data_service = DataService(cfg)
builder = GraphBuilder(cfg, data_service)

subjects = data_service.discover_subjects()
subject_id = subjects[0]
print(f"Running inference on subject: {subject_id}")

graph = builder.build(subject_id)

# Forward pass with attention extraction enabled
with torch.no_grad():
    logits, attention_maps = model.forward_with_attention(graph)
    predictions = logits.argmax(dim=1)

print(f"Predictions: {torch.bincount(predictions, minlength=3)}")
print(f"Attention maps extracted from {len(attention_maps)} layers")
for i, attn in enumerate(attention_maps):
    print(f"  Layer {i}: shape={attn.shape}, "
          f"range=[{attn.min():.4f}, {attn.max():.4f}]")

In [ ]:
# Plot attention heatmaps by layer
# Shows how attention distribution evolves from early to late layers
fig = plot_attention_by_layer(
    attention_maps=attention_maps,
    graph=graph,
    title=f"Attention Heatmaps by Layer - Subject {subject_id}",
)
plt.show()

In [ ]:
# Plot attention aggregated by atlas region
# Shows which anatomical regions receive the most attention
fig = plot_attention_by_region(
    attention_maps=attention_maps,
    graph=graph,
    top_k=20,  # show top-20 most attended regions
    title=f"Top-20 Attended Atlas Regions - Subject {subject_id}",
)
plt.show()

In [ ]:
# Compute and plot PAA (Propagation-based Attention Attribution)
# This propagates attention through all layers to get end-to-end attribution
fig = plot_paa_attribution_map(
    attention_maps=attention_maps,
    graph=graph,
    background_volume=data_service.load_subject(subject_id)["FLAIR"],
    supervoxel_labels=builder.get_supervoxel_labels(subject_id),
    title=f"PAA Attribution Map - Subject {subject_id}",
)
plt.show()

In [ ]:
# Plot stroke core vs penumbra detection
# Compares model predictions with ground truth, colored by PAA attribution
fig = plot_stroke_penumbra_detection(
    graph=graph,
    predictions=predictions,
    attention_maps=attention_maps,
    background_volume=data_service.load_subject(subject_id)["FLAIR"],
    supervoxel_labels=builder.get_supervoxel_labels(subject_id),
    title=f"Stroke Core vs Penumbra Detection - Subject {subject_id}",
)
plt.show()

In [ ]:
# Analyze PAA weight distribution by edge type
# Edge types: intra-region (within same atlas region) vs inter-region
from stroke_gat.models.paa import compute_paa_scores

paa_scores = compute_paa_scores(attention_maps, graph.edge_index)

# Classify edges by type
edge_index = graph.edge_index.numpy()
node_regions = graph.region_labels.numpy() if hasattr(graph, 'region_labels') else None

if node_regions is not None:
    src_regions = node_regions[edge_index[0]]
    dst_regions = node_regions[edge_index[1]]
    is_intra = src_regions == dst_regions

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Intra-region PAA scores
    axes[0].hist(paa_scores[is_intra], bins=50, color="steelblue",
                 edgecolor="white", alpha=0.85, label="Intra-region")
    axes[0].hist(paa_scores[~is_intra], bins=50, color="coral",
                 edgecolor="white", alpha=0.65, label="Inter-region")
    axes[0].set_xlabel("PAA Score", fontsize=12)
    axes[0].set_ylabel("Edge Count", fontsize=12)
    axes[0].set_title("PAA Distribution by Edge Type", fontsize=13)
    axes[0].legend(fontsize=11)

    # Box plot comparison
    data = [paa_scores[is_intra], paa_scores[~is_intra]]
    bp = axes[1].boxplot(data, labels=["Intra-region", "Inter-region"],
                         patch_artist=True)
    bp["boxes"][0].set_facecolor("steelblue")
    bp["boxes"][1].set_facecolor("coral")
    axes[1].set_ylabel("PAA Score", fontsize=12)
    axes[1].set_title("PAA Score Comparison", fontsize=13)

    plt.tight_layout()
    plt.show()

    print(f"Intra-region edges: {is_intra.sum():,} "
          f"(mean PAA: {paa_scores[is_intra].mean():.4f})")
    print(f"Inter-region edges: {(~is_intra).sum():,} "
          f"(mean PAA: {paa_scores[~is_intra].mean():.4f})")
else:
    print("Region labels not available; skipping edge type analysis.")

In [ ]:
# 3D interactive graph with PAA attribution coloring
# Node color intensity reflects the aggregated PAA importance
node_paa = np.zeros(graph.num_nodes)
for i in range(graph.edge_index.shape[1]):
    dst = graph.edge_index[1, i].item()
    node_paa[dst] += paa_scores[i]

# Normalize to [0, 1]
node_paa = (node_paa - node_paa.min()) / (node_paa.max() - node_paa.min() + 1e-8)

fig = plot_graph_3d_interactive(
    graph=graph,
    color_by="custom",
    custom_values=node_paa,
    colorscale="Hot",
    title=f"3D Graph with PAA Attribution - Subject {subject_id}",
    node_size=4,
    edge_alpha=0.05,
)
fig.show()

## Interpretation Guide

### Attention Patterns

- **Layer 1 (early):** Attention is broadly distributed, focusing on local intensity
  differences between neighboring supervoxels.
- **Layer 2 (middle):** Attention becomes more selective, concentrating on boundaries
  between normal and abnormal tissue.
- **Layer 3 (late):** Attention is highly focused on lesion-relevant regions, with
  strong weights on edges connecting penumbra to core regions.

### PAA Attribution

- **High PAA edges** identify the most informative message-passing pathways for
  stroke detection.
- **Inter-region edges** with high PAA often connect the lesion territory to
  surrounding normal tissue, capturing the contrast that defines lesion boundaries.
- **Intra-region edges** with high PAA highlight regions with heterogeneous tissue
  (partial volume effects or mixed lesion/normal tissue).

### Clinical Significance

1. **Lesion delineation:** PAA maps can highlight the boundary between salvageable
   penumbra and irreversible core, which is critical for treatment decisions.
2. **Anatomical context:** Attribution by atlas region reveals which anatomical
   structures are most affected, supporting localization-based clinical assessment.
3. **Model validation:** Clinicians can verify that the model attends to
   physiologically meaningful regions (e.g., diffusion-restricted areas in ADC/TRACE)
   rather than artifacts.